# 02: Fit the Shared OLS Model (Cuisine Group + Name Keywords + Income Quartile)

Reads `data/cleaned_inspections.csv` (produced by `01_data_pull_clean.ipynb`), builds the three predictor groups used throughout the paper -- cuisine group (7 categories), restaurant-name keyword indicators, and zip-code income quartile -- and fits ONE combined OLS model of inspection SCORE on all three. Predictors are unstandardized dummy variables, so each coefficient is a direct SCORE difference vs. its reference group.

Saves two files for the visualization notebooks (`03a`/`03b`/`03c`) to read, so they don't need to reload the data or refit the model themselves:
- `output/shared_model_coefficients.csv` -- one row per predictor (term, coef, ci_low, ci_high, pvalue)
- `data/model_ready_inspections.csv` -- one row per inspection, with score plus the cuisine_group / name_* / income_group columns, for the notebooks' supporting mean-by-group plots

**Takes in:** `data/cleaned_inspections.csv`

**Outputs:** `output/shared_model_coefficients.csv`, `data/model_ready_inspections.csv`

In [1]:
import pandas as pd

import utils

## Load cleaned data and build predictor groups

In [2]:
df = utils.load_cleaned(usecols=["dba", "cuisine_description", "zipcode", "score"])
df = df.dropna(subset=["dba", "cuisine_description", "zipcode", "score"]).copy()
print(f"Rows after dropping missing values: {len(df):,}")

df = utils.add_cuisine_group(df)
df, keyword_cols = utils.add_keyword_indicators(df, name_col="dba")

Loaded 153,452 rows from /Users/ethan/qss20-nyc-restaurant-inspections/data/cleaned_inspections.csv
Rows after dropping missing values: 134,855
Cuisine group counts:
cuisine_group
American & Southern           32810
Asian                         27363
Grab-and-Go                   26784
European                      25770
Latin American & Caribbean    16401
Other/Specialty                3782
Middle Eastern & African       1945
Name: count, dtype: int64


## Merge in zip-code income quartiles

In [3]:
income = utils.get_income_data()  # utils.INCOME_DATA_SOURCE controls real vs. fallback
n_before = len(df)
df = df.merge(income[["zipcode", "income_group"]], on="zipcode", how="inner")
print(f"Rows before income merge: {n_before:,}")
print(f"Rows after income merge (restaurants outside NY ZCTAs with income data are dropped): {len(df):,}")
print(df["income_group"].value_counts().sort_index())

Rows before income merge: 134,855
Rows after income merge (restaurants outside NY ZCTAs with income data are dropped): 131,466
income_group
Q1 (lowest income)          14584
Q2 (lower-middle income)    42914
Q3 (upper-middle income)    32403
Q4 (highest income)         41565
Name: count, dtype: int64


## Build design matrix and fit the shared OLS model

Standard errors are clustered by restaurant chain (normalized `dba` name) rather than treated as independent, since multiple locations of the same chain (e.g. many Panda Express branches) and repeated inspections of the same restaurant aren't truly independent observations -- clustering keeps every inspection in the model (unlike collapsing chains into one row, which would throw away real location-to-location variation) while giving honest uncertainty estimates.

In [4]:
cuisine_dummies = pd.get_dummies(df["cuisine_group"], prefix="cuisine_group", drop_first=True)
income_dummies = pd.get_dummies(df["income_group"], prefix="income_group", drop_first=True)

X = pd.concat([cuisine_dummies, df[keyword_cols], income_dummies], axis=1)
y = df["score"]

print(f"Total predictors: {X.shape[1]} "
      f"({cuisine_dummies.shape[1]} cuisine group, {len(keyword_cols)} name keyword, "
      f"{income_dummies.shape[1]} income group)")

# Cluster by normalized restaurant name so repeated locations/inspections of the same
# chain don't count as independent observations.
chain_id = df["dba"].str.strip().str.upper()
print(f"\n{df.shape[0]:,} inspections across {chain_id.nunique():,} distinct restaurant names")
print("Most common restaurant names (chains) in the sample:")
print(chain_id.value_counts().head(10))

model = utils.fit_shared_model(y, X, cluster_groups=chain_id)

Total predictors: 26 (6 cuisine group, 17 name keyword, 3 income group)

131,466 inspections across 19,763 distinct restaurant names
Most common restaurant names (chains) in the sample:
dba
DUNKIN' DONUTS                    1827
SUBWAY                            1539
MCDONALD'S                        1050
STARBUCKS                         1010
DUNKIN' DONUTS, BASKIN ROBBINS     569
DOMINO'S                           470
KENNEDY FRIED CHICKEN              423
CROWN FRIED CHICKEN                402
BURGER KING                        386
POPEYES LOUISIANA KITCHEN          359
Name: count, dtype: int64


                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     15.16
Date:                Sun, 30 Aug 2026   Prob (F-statistic):           1.07e-66
Time:                        23:28:34   Log-Likelihood:            -4.9415e+05
No. Observations:              131466   AIC:                         9.883e+05
Df Residuals:                  131439   BIC:                         9.886e+05
Df Model:                          26                                         
Covariance Type:              cluster                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

## Save model coefficients and model-ready data for the visualization notebooks

In [5]:
coef_table = utils.save_model_coefficients(model)

model_ready_cols = ["score", "cuisine_group", "income_group"] + keyword_cols
model_ready = df[model_ready_cols].copy()
utils.MODEL_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
model_ready.to_csv(utils.MODEL_READY_PATH, index=False)
print(f"Saved {len(model_ready):,} rows to {utils.MODEL_READY_PATH}")

Saved model coefficients to /Users/ethan/qss20-nyc-restaurant-inspections/output/shared_model_coefficients.csv


Saved 131,466 rows to /Users/ethan/qss20-nyc-restaurant-inspections/data/model_ready_inspections.csv
